# Hypothesis - Explicit and Non Explicit Lyrics Analysis

Analyse whether popularity differs between explicit and non-explicit songs

- Group tracks by explicit status
- Calculate popularity for each group
- Compare the two groups
- Create a suitable visualisation


## Inputs

CSV file used:

spotifydataset_Visualisation.csv


## Outputs

Plots for testing validity of each hypothesis before considering visualisation


## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"



In [1]:
#import libraries
import os
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

#added by me for plotly.express visualisation issue
import nbformat 

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [2]:
#DataFrame variables for EDA
dfSpotify_DataSet_Work = None
dfSpotify_DataSet_Temp = None
dfSpotify_DataSet_Temp1 = None

#stores current directory
strCurrentDir = ""

#other vars
dictDataFrames = dict()
fig = None
axis = None
intCount = 0
objGroups = None
srtTemp = ""
strArtist = ""
strAlbum = ""
strTrack = ""

## Set Current Directory To Base Project Directory

In [3]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project3-Spotify-DirectWrite/Spotify-Music-Trend-Analysis


# Section 2

- Read csv file
- Look at hypothesis validity
- Use plot(s) to validate findings


## Read csv File Into Variable For Processing

In [4]:
#read csv file into DataFrame
dictDataFrames = modETL.funcReadVisualisationFilesReturnDictionary()
dfSpotify_DataSet = dictDataFrames["spotifydataset_Visualisation.csv"]

#create copy of the original DataFrame to work with
dfSpotify_DataSet_Work = dfSpotify_DataSet.copy()

1 csv Files Read Into DataFrames

DataFrames Created:
spotifydataset_Visualisation.csv




# Group Tracks By Explict Status

In [ ]:
#Group tracks by explicit status

#delete pure duplicates and sort by explicit
dfSpotify_DataSet_Temp = (
    dfSpotify_DataSet_Work
    .sort_values(by="popularity", ascending=False)
    .drop_duplicates(
        subset=["artists", "album_name", "track_name"],
        keep="first"
    )
    .drop_duplicates(
        subset=["album_name"],
        keep="first"
    )
    .sort_values(by="explicit", ascending=False)
)



#just show artist, track name and explicit
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["artists", "track_name", "explicit"]]

#show results
display(dfSpotify_DataSet_Temp)

<class 'pandas.core.frame.DataFrame'>


,artists,track_name,explicit
53404,Anne-Marie;The Wild;Aitch,PSYCHO (feat. Aitch) - The Wild Remix,True
39517,YBRE;loufre;Dasco,Hasst mich,True
102521,Lauren Weintraub,Not Like I’m In Love With You,True
94935,Ely Waves;Beautiful Beats,Left Me To Die,True
94332,KillBunk;Dustystaytrue,Outta Time,True
...,...,...,...
15818,Valera;静的 Static,missing us,False
74067,Sandy;Melim,Eu Pra Você,False
202,Masayoshi Yamazaki,"One more time,One more chance",False
652,Aron Wright,Look After You,False


In [19]:
#get count of explict and non explicit songs
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp.groupby("explicit").size().reset_index(name="count")

#show results
dfSpotify_DataSet_Temp1

,explicit,count
0,False,42375
1,True,4214


In [ ]:
#plot count of explicit and non-explicit songs
fig = px.bar(dfSpotify_DataSet_Temp1, x="explicit", y="count", color="explicit",
             title="Count of Songs With Explicit and Non Explicit Lyrics",
             labels={"explicit": "Explicit Lyrics", "count": "Number of Songs"},
             color_discrete_map={0: "blue", 1: "red"})
fig.show()

# Now Check The Popularity

In [20]:
#check popularity of explicit and non-explicit songs

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#group by explicit and get average popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("explicit")["popularity"].mean().reset_index(name="average_popularity")

#show results
dfSpotify_DataSet_Temp

,explicit,average_popularity
0,False,32.878003
1,True,36.942006


In [ ]:
#plot popularity of explicit and non-explicit songs

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#group by explicit and get average popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("explicit")["popularity"].mean().reset_index(name="average_popularity")


fig = px.bar(dfSpotify_DataSet_Temp, x="average_popularity", y="explicit", color="explicit",
             title="Popularity of Songs With Explicit and Non Explicit Lyrics",
             labels={"explicit": "Explicit Lyrics", "count": "Number of Songs"},
             color_discrete_map={0: "blue", 1: "red"})

fig.show()

# Conclusion

We can assume from the sample data that wthere is no clear distinction between popularity of songs with or without explicit lyrics
Which is interesting as common belief would swing towards non explict lyrics being more popular